
**PCA INTERACTIVE EXPLORER**

Explorez la PCA avec différents paramètres et visualisez les résultats en temps réel.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

# Importer votre code PCA
from boed.reduction.methods import PCA, generate_structured_data

Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED


In [7]:
data_path = (repo_root / "data" / "data_pca_10000x201.csv") if repo_root is not None else Path("data/pca/data_pca_10000x201.csv")
if not data_path.exists():
    raise FileNotFoundError(f"CSV introuvable: {data_path} (cwd={Path.cwd()})")

data = pd.read_csv(
    data_path,
    sep=";",
    decimal=","
)
X = data.values


In [ ]:
# ============================================================================
# WIDGETS INTERACTIFS
# ============================================================================

# Paramètres de génération de données
n_samples_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=2000,
    step=100,
    description='N samples:',
    continuous_update=False
)

d_ambient_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description='Dimension:',
    continuous_update=False
)

d_intrinsic_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description='True rank:',
    continuous_update=False
)

noise_slider = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Noise level:',
    continuous_update=False
)

# Paramètres PCA
variance_threshold_slider = widgets.FloatSlider(
    value=0.99,
    min=0.70,
    max=0.9999,
    step=0.01,
    description='Variance %:',
    continuous_update=False
)

method_dropdown = widgets.Dropdown(
    options=['svd', 'eig_cov'],
    value='svd',
    description='Method:',
)

# Bouton de calcul
compute_button = widgets.Button(
    description='Compute PCA',
    button_style='success',
    icon='play'
)

# Output area
output = widgets.Output()

In [9]:


# ============================================================================
# FONCTION DE CALCUL ET VISUALISATION
# ============================================================================

def compute_and_visualize(button=None):
    """Compute PCA et génère les visualisations."""
    
    with output:
        clear_output(wait=True)
        
        print("⏳ Génération des données...")
        
        # Générer données
        # X, true_info = generate_structured_data(
        #     n_samples=n_samples_slider.value,
        #     d_ambient=d_ambient_slider.value,
        #     d_intrinsic=d_intrinsic_slider.value,
        #     noise_level=noise_slider.value,
        #     random_state=42
        # )

        data = pd.read_csv(
            data_path,
            sep=";",
            decimal=","
        )
        X = data.iloc[:,:-1].values
        true_info = None

        
        print(f"✓ Données générées: {X.shape}")
        # print(f"\n⏳ Calcul PCA (méthode: {method_dropdown.value})...")
        
        # PCA
        pca = PCA(variance_threshold=variance_threshold_slider.value)
        pca.fit(X, method=method_dropdown.value)
        
        X_transformed = pca.transform(X)
        X_reconstructed = pca.inverse_transform(X_transformed)
        
        print(f"✓ PCA calculée: {pca.n_components} composantes sélectionnées")
        
        # ====================================================================
        # VISUALISATION
        # ====================================================================
        
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=(
                'Scree Plot',
                'Cumulative Variance',
                'PC1 vs PC2',
                'Reconstruction Error',
                'True vs Estimated',
                'Loadings (First 3 PCs)'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'scatter'}, {'type': 'bar'}, {'type': 'heatmap'}]
            ],
            vertical_spacing=0.12,
            horizontal_spacing=0.1
        )
        
        # SUBPLOT 1 : Scree plot
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(pca.eigenvalues_) + 1)),
                y=pca.eigenvalues_,
                mode='markers+lines',
                marker=dict(size=8, color='blue'),
                name='Eigenvalues'
            ),
            row=1, col=1
        )
        
        fig.add_vline(x=pca.n_components, line_dash='dash', line_color='red',
                      annotation_text=f'Selected: {pca.n_components}', row=1, col=1)
        
        fig.update_yaxes(type='log', title_text='λᵢ', row=1, col=1)
        fig.update_xaxes(title_text='Component', row=1, col=1)
        
        # SUBPLOT 2 : Variance cumulée
        cumsum = np.cumsum(pca.explained_variance_ratio_) * 100
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(cumsum) + 1)),
                y=cumsum,
                mode='lines',
                fill='tozeroy',
                line=dict(color='green', width=3),
                name='Cumulative'
            ),
            row=1, col=2
        )
        
        fig.add_hline(y=variance_threshold_slider.value * 100,
                      line_dash='dash', line_color='red',
                      annotation_text=f'{variance_threshold_slider.value*100:.0f}%',
                      row=1, col=2)
        
        fig.update_yaxes(title_text='Variance (%)', row=1, col=2)
        fig.update_xaxes(title_text='Components', row=1, col=2)
        
        # SUBPLOT 3 : PC1 vs PC2
        fig.add_trace(
            go.Scatter(
                x=X_transformed[:, 0],
                y=X_transformed[:, 1] if X_transformed.shape[1] > 1 else np.zeros_like(X_transformed[:, 0]),
                mode='markers',
                marker=dict(
                    size=5,
                    color=X_transformed[:, 0],
                    colorscale='Viridis',
                    showscale=True
                ),
                name='Samples'
            ),
            row=1, col=3
        )
        
        fig.update_xaxes(title_text='PC1', row=1, col=3)
        fig.update_yaxes(title_text='PC2', row=1, col=3)
        
        # SUBPLOT 4 : Erreur de reconstruction vs rang
        errors = []
        ranks = range(1, min(20, d_ambient_slider.value))
        
        for r in ranks:
            U_r = pca.components_[:, :r]
            X_r = X @ U_r @ U_r.T
            error = np.linalg.norm(X - X_r, 'fro') / np.linalg.norm(X, 'fro')
            errors.append(error)
        
        fig.add_trace(
            go.Scatter(
                x=list(ranks),
                y=errors,
                mode='markers+lines',
                marker=dict(size=8, color='blue'),
                name='Error'
            ),
            row=2, col=1
        )
        
        fig.update_yaxes(type='log', title_text='Relative Error', row=2, col=1)
        fig.update_xaxes(title_text='Rank', row=2, col=1)

        
        
        # SUBPLOT 5 : Comparaison vraies vs estimées
        if true_info is not None:
            true_eigs = true_info['true_eigenvalues']
            n_true = len(true_eigs)
            est_eigs = pca.eigenvalues_[:n_true] / pca.eigenvalues_[:n_true].sum()
            
            x_pos = list(range(1, n_true + 1))
            
            fig.add_trace(
                go.Bar(x=x_pos, y=true_eigs, name='True', marker_color='lightblue'),
                row=2, col=2
            )
            
            fig.add_trace(
                go.Bar(x=x_pos, y=est_eigs, name='Estimated', 
                       marker_color='coral', opacity=0.7),
                row=2, col=2
            )
        
        fig.update_xaxes(title_text='Component', row=2, col=2)
        fig.update_yaxes(title_text='Normalized λ', row=2, col=2)
        
        # SUBPLOT 6 : Loadings heatmap
        n_features_show = min(20, d_ambient_slider.value)
        n_pcs_show = min(3, pca.n_components)
        
        loadings = pca.components_[:n_features_show, :n_pcs_show]
        
        fig.add_trace(
            go.Heatmap(
                z=loadings.T,
                colorscale='RdBu',
                zmid=0,
                x=list(range(1, n_features_show + 1)),
                y=[f'PC{i+1}' for i in range(n_pcs_show)]
            ),
            row=2, col=3
        )
        
        fig.update_xaxes(title_text='Feature', row=2, col=3)
        
        # Layout
        fig.update_layout(
            height=800,
            showlegend=True,
            title_text=f'<b>PCA Analysis</b> (d={d_ambient_slider.value}, r_true={d_intrinsic_slider.value}, noise={noise_slider.value:.2f})'
        )
        
        fig.show()
        
        # Statistiques
        print("\n" + "="*70)
        print("STATISTIQUES")
        print("="*70)
        print(f"Dimension originale     : {d_ambient_slider.value}")
        print(f"Dimension réduite       : {pca.n_components}")
        print(f"Compression             : {d_ambient_slider.value / pca.n_components:.2f}x")
        print(f"Variance capturée       : {cumsum[pca.n_components-1]:.2f}%")
        
        reconstruction_error = np.linalg.norm(X - X_reconstructed, 'fro') / np.linalg.norm(X, 'fro')
        print(f"Erreur reconstruction   : {reconstruction_error:.4e}")
        
        print(f"\nPremiers eigenvalues :")
        for i in range(min(5, len(pca.eigenvalues_))):
            print(f"  λ_{i+1} = {pca.eigenvalues_[i]:.6f} ({pca.explained_variance_ratio_[i]*100:.2f}%)")



In [10]:
# Connecter le bouton
compute_button.on_click(compute_and_visualize)

# ============================================================================
# INTERFACE
# ============================================================================

# Créer l'interface
data_params = widgets.VBox([
    widgets.HTML("<h3>🎲 Data Generation Parameters</h3>"),
    n_samples_slider,
    d_ambient_slider,
    d_intrinsic_slider,
    noise_slider
])

pca_params = widgets.VBox([
    widgets.HTML("<h3>🔧 PCA Parameters</h3>"),
    variance_threshold_slider,
    method_dropdown,
    compute_button
])

controls = widgets.HBox([data_params, pca_params])

# Afficher l'interface
display(widgets.VBox([
    widgets.HTML("<h1>📊 PCA Interactive Explorer</h1>"),
    controls,
    output
]))

# Calcul initial
compute_and_visualize()

In [11]:
"""
NOTEBOOK INTERACTIF : ANALYSE DE VOTRE DATASET 10000×201
=========================================================
"""

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import StandardScaler
import sys
from pathlib import Path

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "boed" / "__init__.py").is_file():
            return p
        if (p / "pyBOED" / "boed" / "__init__.py").is_file():
            return p / "pyBOED"
    return cwd

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for name in list(sys.modules):
    if name == "boed" or name.startswith("boed."):
        del sys.modules[name]

repo_root = PROJECT_ROOT
print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

from boed.reduction.methods import PCA

# ════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES DONNÉES
# ════════════════════════════════════════════════════════════════════════

# Remplacez par le chemin vers votre fichier
DATA_PATH = (repo_root / "data" / "pca" / "data_pca_10000x201.csv") if repo_root is not None else Path("data/pca/data_pca_10000x201.csv")

try:
    # Essayer NumPy
    X_original = np.load(DATA_PATH)
    print(f"✓ Données chargées depuis {DATA_PATH}")
except:
    try:
        # Essayer CSV
        import pandas as pd
        df = pd.read_csv(
            DATA_PATH,
            sep=";",
            decimal=","
        )
        X_original = df.values
        print(f"✓ Données chargées depuis {DATA_PATH} (CSV)")
    except Exception as e:
        print(f"❌ Erreur : {e}")
        print("\\nGénération de données synthétiques pour démo...")
        X_original = np.random.randn(10000, 201)

print(f"Shape : {X_original.shape}")

# ════════════════════════════════════════════════════════════════════════
# WIDGETS INTERACTIFS
# ════════════════════════════════════════════════════════════════════════

normalize_checkbox = widgets.Checkbox(
    value=True,
    description='Normalize data',
    indent=False
)

variance_slider = widgets.FloatSlider(
    value=0.99,
    min=0.50,
    max=0.9999,
    step=0.01,
    description='Variance %:',
    continuous_update=False,
    readout_format='.2%'
)

n_components_manual = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='# Components:',
    continuous_update=False
)

mode_toggle = widgets.ToggleButtons(
    options=['Auto (variance threshold)', 'Manual (fixed rank)'],
    value='Auto (variance threshold)',
    description='Mode:',
    button_style='info'
)

compute_button = widgets.Button(
    description='Run PCA Analysis',
    button_style='success',
    icon='play'
)

output_viz = widgets.Output()
output_stats = widgets.Output()

# ════════════════════════════════════════════════════════════════════════
# FONCTION DE CALCUL
# ════════════════════════════════════════════════════════════════════════

def run_pca_analysis(button=None):
    """Exécute l'analyse PCA."""
    
    with output_viz:
        clear_output(wait=True)
        
        print("⏳ Prétraitement...")
        
        X = X_original.copy()
        
        # Normalisation
        if normalize_checkbox.value:
            scaler = StandardScaler()
            X = scaler.fit_transform(X)
            print("✓ Données normalisées (mean=0, std=1)")
        
        # PCA
        print("\\n⏳ Calcul PCA...")
        
        if mode_toggle.value == 'Auto (variance threshold)':
            pca = PCA(variance_threshold=variance_slider.value)
        else:
            pca = PCA(n_components=n_components_manual.value)
        
        pca.fit(X, method='svd')
        
        print(f"✓ PCA calculée : {pca.n_components} composantes")
        
        X_reduced = pca.transform(X)
        X_recon = pca.inverse_transform(X_reduced)
        
        # ════════════════════════════════════════════════════════════════
        # VISUALISATION
        # ════════════════════════════════════════════════════════════════
        
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'Scree Plot (Eigenvalues)',
                'Cumulative Variance',
                'PC1 vs PC2 Projection',
                'First 3 PC Modes (spatial)',
                'Reconstruction Error vs Rank',
                'Sample Reconstruction (first sample)'
            ),
            specs=[
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'scatter'}, {'type': 'scatter'}],
                [{'type': 'scatter'}, {'type': 'scatter'}]
            ],
            vertical_spacing=0.12,
            horizontal_spacing=0.15
        )
        
        # SUBPLOT 1 : Scree plot
        n_show = min(50, len(pca.eigenvalues_))
        
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, n_show + 1)),
                y=pca.eigenvalues_[:n_show],
                mode='markers+lines',
                marker=dict(size=6, color='blue'),
                name='Eigenvalues'
            ),
            row=1, col=1
        )
        
        fig.add_vline(x=pca.n_components, line_dash='dash', line_color='red',
                      annotation_text=f'r={pca.n_components}', row=1, col=1)
        
        fig.update_yaxes(type='log', title_text='λᵢ', row=1, col=1)
        fig.update_xaxes(title_text='Component i', row=1, col=1)
        
        # SUBPLOT 2 : Cumulative variance
        cumsum = np.cumsum(pca.explained_variance_ratio_) * 100
        
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(cumsum) + 1)),
                y=cumsum,
                mode='lines',
                fill='tozeroy',
                line=dict(color='green', width=3),
                name='Cumulative'
            ),
            row=1, col=2
        )
        
        if mode_toggle.value == 'Auto (variance threshold)':
            fig.add_hline(y=variance_slider.value * 100, 
                         line_dash='dash', line_color='red', row=1, col=2)
        
        fig.update_yaxes(title_text='Variance (%)', row=1, col=2)
        fig.update_xaxes(title_text='Components', row=1, col=2)
        
        # SUBPLOT 3 : Projection 2D
        if pca.n_components >= 2:
            fig.add_trace(
                go.Scatter(
                    x=X_reduced[:, 0],
                    y=X_reduced[:, 1],
                    mode='markers',
                    marker=dict(
                        size=3,
                        color=X_reduced[:, 0],
                        colorscale='Viridis',
                        showscale=True,
                        colorbar=dict(x=0.46, len=0.3, title='PC1')
                    ),
                    name='Samples'
                ),
                row=2, col=1
            )
            
            fig.update_xaxes(title_text='PC1', row=2, col=1)
            fig.update_yaxes(title_text='PC2', row=2, col=1)
        
        # SUBPLOT 4 : Modes spatiaux (3 premiers)
        space_axis = np.arange(1, 202)  # 201 points
        
        for i in range(min(3, pca.n_components)):
            # Mode i = i-ème vecteur propre
            mode = pca.components_[:, i]
            
            fig.add_trace(
                go.Scatter(
                    x=space_axis,
                    y=mode,
                    mode='lines',
                    name=f'PC{i+1}',
                    line=dict(width=2)
                ),
                row=2, col=2
            )
        
        fig.update_xaxes(title_text='Spatial index (1-201)', row=2, col=2)
        fig.update_yaxes(title_text='Mode amplitude', row=2, col=2)
        
        # SUBPLOT 5 : Erreur vs rang
        errors = []
        ranks = range(1, min(100, len(pca.eigenvalues_)))
        
        for r in ranks:
            residual_energy = pca.eigenvalues_[r:].sum()
            total_energy = pca.eigenvalues_.sum()
            error = np.sqrt(residual_energy / total_energy)
            errors.append(error)
        
        fig.add_trace(
            go.Scatter(
                x=list(ranks),
                y=errors,
                mode='lines',
                line=dict(color='red', width=2),
                name='Theoretical error'
            ),
            row=3, col=1
        )
        
        fig.update_yaxes(type='log', title_text='Relative Error', row=3, col=1)
        fig.update_xaxes(title_text='Rank', row=3, col=1)
        
        # SUBPLOT 6 : Reconstruction d'un échantillon
        sample_idx = 0
        
        fig.add_trace(
            go.Scatter(
                x=space_axis,
                y=X[sample_idx],
                mode='lines',
                name='Original',
                line=dict(color='blue', width=2)
            ),
            row=3, col=2
        )
        
        fig.add_trace(
            go.Scatter(
                x=space_axis,
                y=X_recon[sample_idx],
                mode='lines',
                name=f'Reconstructed (r={pca.n_components})',
                line=dict(color='red', width=2, dash='dash')
            ),
            row=3, col=2
        )
        
        fig.update_xaxes(title_text='Spatial index', row=3, col=2)
        fig.update_yaxes(title_text='Value', row=3, col=2)
        
        # Layout
        fig.update_layout(
            height=1200,
            showlegend=True,
            title_text=f'<b>PCA Analysis : 10000 samples × 201 features</b>'
        )
        
        fig.show()
    
    # ════════════════════════════════════════════════════════════════════
    # STATISTIQUES
    # ════════════════════════════════════════════════════════════════════
    
    with output_stats:
        clear_output(wait=True)
        
        print("="*70)
        print("RÉSULTATS")
        print("="*70)
        
        print(f"\\nDonnées originales     : {X_original.shape}")
        print(f"Données réduites       : {X_reduced.shape}")
        print(f"Compression            : {201 / pca.n_components:.2f}x")
        print(f"Variance capturée      : {cumsum[pca.n_components-1]:.4f}%")
        
        # Erreur reconstruction
        recon_error = np.linalg.norm(X - X_recon, 'fro') / np.linalg.norm(X, 'fro')
        print(f"Erreur reconstruction  : {recon_error:.6f}")
        
        print(f"\\n10 premiers eigenvalues :")
        print("-"*70)
        for i in range(min(10, len(pca.eigenvalues_))):
            print(f"λ_{i+1:2d} = {pca.eigenvalues_[i]:12.6e}  "
                  f"({pca.explained_variance_ratio_[i]*100:6.2f}%)")
        
        # Gap spectral
        if len(pca.eigenvalues_) > 1:
            ratios = pca.eigenvalues_[:-1] / pca.eigenvalues_[1:]
            max_gap_idx = np.argmax(ratios)
            
            print(f"\\nGap spectral maximal :")
            print(f"  Entre λ_{max_gap_idx+1} et λ_{max_gap_idx+2}")
            print(f"  Ratio : {ratios[max_gap_idx]:.2f}")
            print(f"  → Dimension intrinsèque suggérée : {max_gap_idx + 1}")
        
        # Informations stockage
        original_size = X_original.nbytes / 1024 / 1024  # MB
        reduced_size = X_reduced.nbytes / 1024 / 1024
        
        print(f"\\nStockage :")
        print(f"  Original : {original_size:.2f} MB")
        print(f"  Réduit   : {reduced_size:.2f} MB")
        print(f"  Économie : {(1 - reduced_size/original_size)*100:.1f}%")


compute_button.on_click(run_pca_analysis)

# ════════════════════════════════════════════════════════════════════════
# INTERFACE
# ════════════════════════════════════════════════════════════════════════

controls = widgets.VBox([
    widgets.HTML("<h2>⚙️ PCA Parameters</h2>"),
    normalize_checkbox,
    mode_toggle,
    variance_slider,
    n_components_manual,
    widgets.HTML("<br>"),
    compute_button
])

display(widgets.VBox([
    widgets.HTML(f"<h1>📊 PCA Analysis : Dataset 10000×201</h1>"),
    widgets.HTML(f"<p><b>Data shape:</b> {X_original.shape[0]} samples × {X_original.shape[1]} features</p>"),
    controls,
    widgets.HTML("<h2>📈 Visualizations</h2>"),
    output_viz,
    widgets.HTML("<h2>📊 Statistics</h2>"),
    output_stats
]))

# Calcul initial
run_pca_analysis()

Using PROJECT_ROOT=/home/mdoumbou/Documents/Biblio_thèse/pyBOED
❌ Erreur : [Errno 2] No such file or directory: '/home/mdoumbou/Documents/Biblio_thèse/pyBOED/data/pca/data_pca_10000x201.csv'
\nGénération de données synthétiques pour démo...
Shape : (10000, 201)
